# 连接到 LangGraph Platform 部署

## 部署创建

我们刚刚为模块5中的 `task_maistro` 应用创建了一个[部署](https://langchain-ai.github.io/langgraph/how-tos/deploy-self-hosted/#how-to-do-a-self-hosted-deployment-of-langgraph)。

* 我们使用[LangGraph CLI](https://langchain-ai.github.io/langgraph/concepts/langgraph_cli/#commands)为 LangGraph Server 构建了一个包含我们 `task_maistro` 图的 Docker 镜像。
* 我们使用提供的 `docker-compose.yml` 文件基于定义的服务创建了三个独立的容器：
    * `langgraph-redis`：使用官方 Redis 镜像创建新容器。
    * `langgraph-postgres`：使用官方 Postgres 镜像创建新容器。
    * `langgraph-api`：使用我们预构建的 `task_maistro` Docker 镜像创建新容器。

```
$ cd module-6/deployment
$ docker compose up
```

运行后，我们可以通过以下方式访问部署：
      
* API：http://localhost:8123
* 文档：http://localhost:8123/docs
* LangGraph Studio：https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:8123

![langgraph-platform-high-level.png](attachment:3a5ede4f-7a62-4e05-9e44-301465ca8555.png)

## 使用 API

LangGraph Server 暴露了[许多 API 端点](https://github.com/langchain-ai/agent-protocol)用于与部署的代理交互。

我们可以将[这些端点分组为几个常见的代理需求](https://github.com/langchain-ai/agent-protocol)：

* **Runs**：原子代理执行
* **Threads**：多轮交互或人在回路中
* **Store**：长期记忆

我们可以直接在[API 文档](http://localhost:8123/docs#tag/thread-runs)中测试请求。

## SDK

[LangGraph SDKs](https://langchain-ai.github.io/langgraph/concepts/sdk/)（Python 和 JS）提供了一个开发者友好的接口来与上面展示的 LangGraph Server API 交互。

In [ ]:
%%capture --no-stderr
%pip install -U langgraph_sdk

In [ ]:
from langgraph_sdk import get_client

# 通过 SDK 连接
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

## 远程图

如果你在 LangGraph 库中工作，[Remote Graph](https://langchain-ai.github.io/langgraph/how-tos/use-remote-graph/) 也是直接连接到图的有用方式。

In [ ]:
%%capture --no-stderr
%pip install -U langchain_openai langgraph langchain_core

In [ ]:
from langgraph.pregel.remote import RemoteGraph
from langchain_core.messages import convert_to_messages
from langchain_core.messages import HumanMessage, SystemMessage

# 通过远程图连接
url = "http://localhost:8123"
graph_name = "task_maistro" 
remote_graph = RemoteGraph(graph_name, url=url)

## Runs

一个"run"代表你的图的[单次执行](https://github.com/langchain-ai/agent-protocol?tab=readme-ov-file#runs-atomic-agent-executions)。每次客户端发出请求时：

1. HTTP worker 生成唯一的 run ID
2. 这个 run 及其结果存储在 PostgreSQL 中
3. 你可以查询这些 runs 来：
   - 检查它们的状态
   - 获取它们的结果
   - 跟踪执行历史

你可以在[这里](https://langchain-ai.github.io/langgraph/how-tos/#runs)看到各种类型 runs 的完整操作指南集。

让我们看看我们可以用 runs 做的一些有趣的事情。

### 后台 Runs

LangGraph 服务器支持两种类型的 runs：

* `即发即忘` - 在后台启动一个 run，但不等待它完成
* `等待回复（阻塞或轮询）` - 启动一个 run 并等待/流式传输其输出

当处理长时间运行的代理时，后台 runs 和轮询非常有用。

让我们[看看](https://langchain-ai.github.io/langgraph/cloud/how-tos/background_run/#check-runs-on-thread)这是如何工作的：

In [ ]:
# 创建一个线程
thread = await client.threads.create()
thread

In [ ]:
# 检查线程上的任何现有 runs
thread = await client.threads.create()
runs = await client.runs.list(thread["thread_id"])
print(runs)

In [ ]:
# 确保我们已经创建了一些待办事项并将其保存到我的 user_id
user_input = "添加一个待办事项，在下周末前完成预订去香港的旅行。另外，添加一个待办事项，回电话给父母讨论感恩节计划。"
config = {"configurable": {"user_id": "Test"}}
graph_name = "task_maistro" 
run = await client.runs.create(thread["thread_id"], graph_name, input={"messages": [HumanMessage(content=user_input)]}, config=config)

In [ ]:
# 启动一个新线程和一个新 run
thread = await client.threads.create()
user_input = "给我所有待办事项的摘要。"
config = {"configurable": {"user_id": "Test"}}
graph_name = "task_maistro" 
run = await client.runs.create(thread["thread_id"], graph_name, input={"messages": [HumanMessage(content=user_input)]}, config=config)

In [ ]:
# 检查 run 状态
print(await client.runs.get(thread["thread_id"], run["run_id"]))

我们可以看到它有 `'status': 'pending'`，因为它仍在运行。

如果我们想等到 run 完成，使其成为阻塞 run 怎么办？

我们可以使用 `client.runs.join` 等待 run 完成。

这确保在当前 run 在线程上完成之前不会启动新的 runs。

In [ ]:
# 等待 run 完成
await client.runs.join(thread["thread_id"], run["run_id"])
print(await client.runs.get(thread["thread_id"], run["run_id"]))

现在 run 有 `'status': 'success'`，因为它已经完成。

### 流式 Runs

每次客户端发出流式请求时：

1. HTTP worker 生成唯一的 run ID
2. Queue worker 开始处理 run
3. 在执行过程中，Queue worker 向 Redis 发布更新
4. HTTP worker 订阅来自 Redis 的此 run 的更新，并将其返回给客户端

这启用了流式传输！

我们在之前的模块中介绍了[流式传输](https://langchain-ai.github.io/langgraph/how-tos/#streaming_1)，但让我们挑选一种方法——流式令牌——来突出显示。

当处理可能需要很长时间才能完成的生产代理时，将令牌流回客户端特别有用。

我们使用 `stream_mode="messages-tuple"` [流式令牌](https://langchain-ai.github.io/langgraph/cloud/how-tos/stream_messages/#setup)。

In [ ]:
user_input = "我应该首先关注哪个待办事项。"
async for chunk in client.runs.stream(thread["thread_id"], 
                                      graph_name, 
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      config=config,
                                      stream_mode="messages-tuple"):

    if chunk.event == "messages":
        print("".join(data_item['content'] for data_item in chunk.data if 'content' in data_item), end="", flush=True)

## Threads

而 run 只是图的单次执行，thread 支持*多轮*交互。

当客户端使用 `thread_id` 进行图执行时，服务器将把 run 中的所有[检查点](https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpoints)（步骤）保存到 Postgres 数据库中的线程。

服务器允许我们[检查创建的线程的状态](https://langchain-ai.github.io/langgraph/cloud/how-tos/check_thread_status/)。

### 检查线程状态

此外，我们可以轻松访问保存到任何特定线程的状态[检查点](https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpoints)。

In [ ]:
thread_state = await client.threads.get_state(thread['thread_id'])
for m in convert_to_messages(thread_state['values']['messages']):
    m.pretty_print()

### 复制线程

我们也可以[复制](https://langchain-ai.github.io/langgraph/cloud/how-tos/copy_threads/)（即"分叉"）现有线程。

这将保留现有线程的历史，但允许我们创建不影响原始线程的独立 runs。

In [ ]:
# 复制线程
copied_thread = await client.threads.copy(thread['thread_id'])

In [ ]:
# 检查复制线程的状态
copied_thread_state = await client.threads.get_state(copied_thread['thread_id'])
for m in convert_to_messages(copied_thread_state['values']['messages']):
    m.pretty_print()

### 人在回路中

我们在模块3中介绍了[人在回路中](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/)，服务器支持我们讨论的所有人在回路中功能。

作为示例，[我们可以从任何先前的检查点搜索、编辑和继续图执行](https://langchain-ai.github.io/langgraph/concepts/persistence/#capabilities)。

In [ ]:
# 获取线程的历史
states = await client.threads.get_history(thread['thread_id'])

# 选择一个状态更新来分叉
to_fork = states[-2]
to_fork['values']

In [ ]:
to_fork['values']['messages'][0]['id']

In [ ]:
to_fork['next']

In [ ]:
to_fork['checkpoint_id']

让我们编辑状态。记住我们在 `messages` 上的 reducer 是如何工作的：

* 它会追加，除非我们提供消息 ID。
* 我们提供消息 ID 来覆盖消息，而不是追加到状态！

In [ ]:
forked_input = {"messages": HumanMessage(content="给我一个需要在下周内完成的所有待办事项摘要。",
                                         id=to_fork['values']['messages'][0]['id'])}

# 更新状态，在线程中创建新检查点
forked_config = await client.threads.update_state(
    thread["thread_id"],
    forked_input,
    checkpoint_id=to_fork['checkpoint_id']
)

In [ ]:
# 从线程中的新检查点运行图
async for chunk in client.runs.stream(thread["thread_id"], 
                                      graph_name, 
                                      input=None,
                                      config=config,
                                      checkpoint_id=forked_config['checkpoint_id'],
                                      stream_mode="messages-tuple"):

    if chunk.event == "messages":
        print("".join(data_item['content'] for data_item in chunk.data if 'content' in data_item), end="", flush=True)

## 跨线程记忆

在模块5中，我们介绍了如何使用[LangGraph 记忆 `store`](https://langchain-ai.github.io/langgraph/concepts/persistence/#memory-store)来保存跨线程的信息。

我们部署的图 `task_maistro` 使用 `store` 来保存信息——比如待办事项——命名空间为 `user_id`。

我们的部署包括一个 Postgres 数据库，它存储这些长期（跨线程）记忆。

有几种方法可用于使用 LangGraph SDK [与我们部署中的 store 交互](https://langchain-ai.github.io/langgraph/cloud/reference/sdk/python_sdk_ref/#langgraph_sdk.client.StoreClient)。

### 搜索项目

`task_maistro` 图使用 `store` 来保存默认命名空间为（`todo`、`todo_category`、`user_id`）的待办事项。

`todo_category` 默认设置为 `general`（如你在 `deployment/configuration.py` 中看到的）。

我们可以简单地提供这个元组来搜索所有待办事项。

In [ ]:
items = await client.store.search_items(
    ("todo", "general", "Test"),
    limit=5,
    offset=0
)
items['items']

### 添加项目

在我们的图中，我们调用 `put` 来向 store 添加项目。

如果我们想在图外直接向 store 添加项目，我们可以使用 SDK 的 [put](https://langchain-ai.github.io/langgraph/cloud/reference/sdk/python_sdk_ref/#langgraph_sdk.client.StoreClient.put_item)。

In [ ]:
from uuid import uuid4
await client.store.put_item(
    ("testing", "Test"),
    key=str(uuid4()),
    value={"todo": "测试 SDK put_item"},
)

In [ ]:
items = await client.store.search_items(
    ("testing", "Test"),
    limit=5,
    offset=0
)
items['items']

### 删除项目

我们可以使用 SDK 通过键从 store 中[删除项目](https://langchain-ai.github.io/langgraph/cloud/reference/sdk/python_sdk_ref/#langgraph_sdk.client.StoreClient.delete_item)。

In [ ]:
[item['key'] for item in items['items']]

In [ ]:
await client.store.delete_item(
       ("testing", "Test"),
        key='3de441ba-8c79-4beb-8f52-00e4dcba68d4',
    )

In [ ]:
items = await client.store.search_items(
    ("testing", "Test"),
    limit=5,
    offset=0
)
items['items']